# Heart Disease Prediction - Final Assignment

Diploma in Data Science & Machine Learning

This notebook demonstrates a complete end-to-end machine learning and deep learning workflow for heart disease prediction.

## 1. Import Libraries

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import TARGET_COL, RANDOM_STATE
from src.data_loader import load_dataset
from src.train_and_evaluate import (
    preprocess_input,
    train_ml_models,
    train_ann,
    save_history_plot,
    save_roc_curves,
    ensure_output_dirs,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
np.random.seed(RANDOM_STATE)

## 2. Load Dataset

In [ ]:
ensure_output_dirs()
df = load_dataset()
df = preprocess_input(df)
df.head()

In [ ]:
print("Shape:", df.shape)
print("Missing Values:\n", df.isna().sum())
print("Class Distribution:\n", df[TARGET_COL].value_counts(normalize=True))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
numeric_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET_COL, palette="Set2")
plt.title("Target Class Distribution")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.tight_layout()
plt.show()

## 4. Train/Test Split

In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

## 5. Train Machine Learning Models (with Hyperparameter Tuning)

In [ ]:
ml_summary, ml_details, ml_roc_data = train_ml_models(X_train, X_test, y_train, y_test)
ml_summary

## 6. Train Deep Learning Model (ANN)

In [ ]:
ann_metrics, ann_history, ann_roc = train_ann(X_train, X_test, y_train, y_test)
save_history_plot(ann_history)
ann_metrics

## 7. Consolidated Model Comparison

In [ ]:
all_roc_data = dict(ml_roc_data)
all_roc_data["Artificial Neural Network"] = ann_roc
save_roc_curves(all_roc_data)

ann_row = pd.DataFrame([
    {
        "model": ann_metrics["model"],
        "accuracy": ann_metrics["accuracy"],
        "precision": ann_metrics["precision"],
        "recall": ann_metrics["recall"],
        "f1_score": ann_metrics["f1_score"],
        "roc_auc": ann_metrics["roc_auc"],
    }
])

comparison = pd.concat([ml_summary, ann_row], ignore_index=True).sort_values("roc_auc", ascending=False)
comparison

In [ ]:
best_model = comparison.iloc[0]["model"]
print(f"Best model based on ROC-AUC: {best_model}")

## 8. View Saved Artifacts

In [ ]:
print("Models:")
for p in sorted((project_root / "models").glob("*")):
    print(" -", p.name)

print("\nReports:")
for p in sorted((project_root / "reports").glob("*")):
    print(" -", p.name)

## 9. Conclusion

This notebook completes the project workflow:
- Data loading and quality checks
- EDA and visualization
- Preprocessing and feature transformation
- Model training (ML + ANN)
- Hyperparameter tuning
- Model evaluation and comparison
- Artifact saving for deployment/inference